In [65]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from linearmodels import PanelOLS, RandomEffects, PooledOLS
from linearmodels.panel import compare
from scipy import stats
import statsmodels.api as sm
from statsmodels.stats.diagnostic import het_breuschpagan

ModuleNotFoundError: No module named 'linearmodels'

In [ ]:
# Set display options
pd.set_option('display.max_rows', 10)
pd.set_option('display.float_format', '{:.2f}'.format)

# Create the toy dataset
data = pd.DataFrame({
    'company': ['A', 'A', 'A', 'A', 'A',
                'B', 'B', 'B', 'B', 'B',
                'C', 'C', 'C', 'C', 'C',
                'D', 'D', 'D', 'D', 'D',
                'E', 'E', 'E', 'E', 'E'],
    'year': [2019, 2020, 2021, 2022, 2023] * 5,
    'profit_margin': [10.2, 10.5, 11.0, 10.8, 11.5,
                      15.1, 15.5, 16.0, 16.2, 17.0,
                      5.5, 5.8, 6.1, 6.0, 6.8,
                      20.1, 20.5, 21.0, 21.2, 22.0,
                      8.1, 8.5, 9.0, 8.8, 9.5],
    'rd_spending': [4.1, 4.3, 4.5, 4.4, 4.8,
                    2.1, 2.2, 2.5, 2.6, 3.0,
                    8.1, 8.3, 8.5, 8.4, 9.0,
                    1.1, 1.2, 1.5, 1.6, 2.0,
                    6.1, 6.3, 6.5, 6.4, 6.8]
})

# Display the first few rows
print("Dataset Preview:")
print(data.head(10))
print("\nDataset Info:")
print(f"Number of observations: {len(data)}")
print(f"Number of companies: {data['company'].nunique()}")
print(f"Years: {data['year'].min()} to {data['year'].max()}")

In [ ]:
# IMPORTANT: Set the multi-index for panel data
# The index must be (entity, time) for linearmodels
data = data.set_index(['company', 'year'])
data = data.sort_index()

print("\nData with Panel Index:")
print(data.head(10))
print("\nIndex structure:")
print(data.index)

In [ ]:
# Create visualization to understand the data
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Time trends by company
for company in data.index.get_level_values('company').unique():
    company_data = data.xs(company, level='company')
    axes[0, 0].plot(company_data.index, company_data['profit_margin'], 
                    marker='o', label=f'Company {company}')
axes[0, 0].set_title('Profit Margin Trends by Company')
axes[0, 0].set_xlabel('Year')
axes[0, 0].set_ylabel('Profit Margin (%)')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# 2. R&D spending trends
for company in data.index.get_level_values('company').unique():
    company_data = data.xs(company, level='company')
    axes[0, 1].plot(company_data.index, company_data['rd_spending'], 
                    marker='s', label=f'Company {company}')
axes[0, 1].set_title('R&D Spending Trends by Company')
axes[0, 1].set_xlabel('Year')
axes[0, 1].set_ylabel('R&D Spending (% of Revenue)')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# 3. Scatter plot: R&D vs Profit Margin
axes[1, 0].scatter(data['rd_spending'], data['profit_margin'], alpha=0.7)
axes[1, 0].set_xlabel('R&D Spending (% of Revenue)')
axes[1, 0].set_ylabel('Profit Margin (%)')
axes[1, 0].set_title('R&D Spending vs Profit Margin')
axes[1, 0].grid(True, alpha=0.3)

# Add a regression line
z = np.polyfit(data['rd_spending'], data['profit_margin'], 1)
p = np.poly1d(z)
axes[1, 0].plot(data['rd_spending'].sort_values(), 
                p(data['rd_spending'].sort_values()), 
                "r--", alpha=0.8, label='Trend line')
axes[1, 0].legend()

# 4. Boxplot by company
data.groupby('company')['profit_margin'].mean().sort_values().plot(
    kind='bar', ax=axes[1, 1], color='skyblue', edgecolor='black')
axes[1, 1].set_title('Average Profit Margin by Company')
axes[1, 1].set_xlabel('Company')
axes[1, 1].set_ylabel('Average Profit Margin (%)')
axes[1, 1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

# Summary statistics
print("\nSummary Statistics:")
print(data.describe())

# Between and within variation
print("\nVariation Decomposition:")
print("Between-company variation (standard deviation of company means):")
company_means = data.groupby('company').mean()
print(company_means.std())

print("\nWithin-company variation (average of within-company std devs):")
within_std = data.groupby('company').std().mean()
print(within_std)

In [39]:
data = pd.read_excel(r'data\WDI_GFDI_finance.xlsx')

In [41]:
data = data.pivot_table(index=['Country Name', 'Year'],
                       columns=['Indicator Name'],
                                 values=['Value'],
                                 aggfunc='sum')#.to_excel(r'wdi_piovt.xlsx')

In [57]:
data.columns = data.columns.droplevel(0) 

In [59]:
data

Indicator Name      Bank deposits to GDP (%)  \
Country Name  Year                             
Australia     1994                  53.49370   
              1995                  54.85179   
              1996                  57.46983   
              1997                  58.39319   
              1998                  60.02831   
...                                      ...   
United States 2016                  82.06911   
              2017                  82.21514   
              2018                  80.99707   
              2019                  84.52532   
              2020                 101.21690   

Indicator Name      Domestic credit to private sector (% of GDP)  \
Country Name  Year                                                 
Australia     1994                                     66.769015   
              1995                                     69.773749   
              1996                                     71.496769   
              1997                                     75.318730   
              1998                                     79.110667   
...                                                          ...   
United States 2016                                    186.327051   
              2017                                    194.090997   
              2018                                    183.054002   
              2019                                    194.735259   
              2020                                    218.335612   

Indicator Name      GDP (constant 2015 US$)  \
Country Name  Year                            
Australia     1994             6.813110e+11   
              1995             7.077863e+11   
              1996             7.351344e+11   
              1997             7.639020e+11   
              1998             7.995244e+11   
...                                     ...   
United States 2016             1.862789e+13   
              2017             1.908569e+13   
              2018             1.965187e+13   
              2019             2.015964e+13   
              2020             1.972358e+13   

Indicator Name      Market capitalization of listed domestic companies (% of GDP)  \
Country Name  Year                                                                  
Australia     1994                                          67.800709               
              1995                                          66.605367               
              1996                                          77.858142               
              1997                                          67.891791               
              1998                                          82.299107               
...                                                               ...               
United States 2016                                         145.452418               
              2017                                         163.780010               
              2018                                         147.344853               
              2019                                         158.243992               
              2020                                         194.669183               

Indicator Name      Stocks traded, total value (% of GDP)  
Country Name  Year                                         
Australia     1994                              30.850800  
              1995                              26.672029  
              1996                              36.119829  
              1997                              34.480500  
              1998                              38.915077  
...                                                   ...  
United States 2016                             223.725204  
              2017                             202.863933  
              2018                             237.519509  
              2019                             168.720262  
              2020                             192.4

In [63]:
# Prepare variables
# Note: linearmodels requires explicit addition of constant for some models
X = data[['Bank deposits to GDP (%)', 'Domestic credit to private sector (% of GDP)', 'Stocks traded, total value (% of GDP)', 'Market capitalization of listed domestic companies (% of GDP)']]
y = data['GDP (constant 2015 US$)']

# Add constant for models that need it
X_with_const = sm.add_constant(X)

print("="*60)
print("PANEL DATA REGRESSION RESULTS")
print("="*60)

# 1. POOLED OLS (ignores panel structure)
print("\n1. POOLED OLS MODEL")
print("-"*40)
pooled_model = PooledOLS(y, X_with_const)
pooled_results = pooled_model.fit(cov_type='robust')
print(pooled_results)

# 2. FIXED EFFECTS MODEL (entity effects)
print("\n2. FIXED EFFECTS MODEL")
print("-"*40)
# entity_effects=True includes fixed effects for each company
fe_model = PanelOLS(y, X, entity_effects=True)
fe_results = fe_model.fit(cov_type='clustered', cluster_entity=True)
print(fe_results)

# 3. RANDOM EFFECTS MODEL
print("\n3. RANDOM EFFECTS MODEL")
print("-"*40)
re_model = RandomEffects(y, X_with_const)
re_results = re_model.fit(cov_type='robust')
print(re_results)

# 4. TWO-WAY FIXED EFFECTS (entity + time effects)
print("\n4. TWO-WAY FIXED EFFECTS MODEL")
print("-"*40)
twfe_model = PanelOLS(y, X, entity_effects=True, time_effects=True)
twfe_results = twfe_model.fit(cov_type='clustered', cluster_entity=True)
print(twfe_results)

NameError: name 'sm' is not defined

In [ ]:
fe_model = PanelOLS(y, X)
fe_results = fe_model.fit()
print(fe_results)

In [ ]:
fe_model = PanelOLS(y, X, time_effects=True)
fe_results = fe_model.fit(cov_type='clustered', cluster_entity=True)
print(fe_results)

In [ ]:
fe_model = PanelOLS(y, X, entity_effects=True)
fe_results = fe_model.fit(cov_type='clustered', cluster_entity=True)
print(fe_results)